# AIC 2025 — GPU TransNet V2 Shot Segmentation & Smart Merge

Fast GPU batch video segmentation using **TransNet V2** (3D-CNN on CUDA) and **`smart_merge_shots`**.

- **Runtime**: ~5-7s per video (~20 min for remaining 272 videos on T4 GPU).
- **Output**: Zipped `shot_boundaries.zip` containing `{video_id}.json` files ready for ReCap.

In [ ]:
# 1. Install dependencies
!pip install -q transnetv2-pytorch tqdm opencv-python ffmpeg-python

In [ ]:
import os
import sys
import csv
import json
import glob
import shutil
from pathlib import Path
import numpy as np
import cv2
import torch
from tqdm.auto import tqdm
from transnetv2_pytorch import TransNetV2

# 1. Hardware Detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 2. Path Configurations
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/shot_boundaries")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Settings
MIN_SHOT_DURATION = 10.0      # Min duration for smart merge (seconds)
START_VIDEO_ID = "L26_V128"    # Start processing from this video (or None for all)
END_VIDEO_ID = None            # Optional end video (e.g. 'L26_V399' or None)

In [ ]:
def to_seconds(val) -> float:
    """Convert timestamps (str or numeric) to float seconds."""
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        if ":" in val:
            parts = val.split(":")
            if len(parts) == 3:
                return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
            elif len(parts) == 2:
                return float(parts[0]) * 60 + float(parts[1])
        return float(val)
    return float(val)

def smart_merge_shots(shots: list, min_duration: float = 10.0) -> list:
    """
    Greedy shortest-neighbor merge from making_shot:
    Iteratively merges the shortest shot into its shorter neighbor until all shots >= min_duration.
    """
    if not shots:
        return []

    cleaned_shots = []
    for s in shots:
        st = to_seconds(s["start_time"])
        et = to_seconds(s["end_time"])
        cleaned_shots.append({
            "start_time": st,
            "end_time": et,
            "duration": et - st,
            "start_frame": int(s.get("start_frame", 0)),
            "end_frame": int(s.get("end_frame", 0))
        })

    while True:
        if len(cleaned_shots) <= 1:
            break

        min_shot_idx = min(range(len(cleaned_shots)), key=lambda i: cleaned_shots[i]["duration"])
        min_shot = cleaned_shots[min_shot_idx]

        if min_shot["duration"] >= min_duration:
            break

        left_idx = min_shot_idx - 1 if min_shot_idx > 0 else None
        right_idx = min_shot_idx + 1 if min_shot_idx < len(cleaned_shots) - 1 else None

        if left_idx is not None and right_idx is not None:
            if cleaned_shots[left_idx]["duration"] <= cleaned_shots[right_idx]["duration"]:
                target_idx = left_idx
            else:
                target_idx = right_idx
        elif left_idx is not None:
            target_idx = left_idx
        else:
            target_idx = right_idx

        if target_idx < min_shot_idx:
            # Merge into left
            cleaned_shots[target_idx]["end_time"] = min_shot["end_time"]
            cleaned_shots[target_idx]["end_frame"] = min_shot["end_frame"]
            cleaned_shots[target_idx]["duration"] = cleaned_shots[target_idx]["end_time"] - cleaned_shots[target_idx]["start_time"]
            cleaned_shots.pop(min_shot_idx)
        else:
            # Merge into right
            cleaned_shots[target_idx]["start_time"] = min_shot["start_time"]
            cleaned_shots[target_idx]["start_frame"] = min_shot["start_frame"]
            cleaned_shots[target_idx]["duration"] = cleaned_shots[target_idx]["end_time"] - cleaned_shots[target_idx]["start_time"]
            cleaned_shots.pop(min_shot_idx)

    return cleaned_shots

In [ ]:
# Initialize TransNetV2 model on GPU with fallback
try:
    print("Initializing TransNetV2 on GPU...")
    model = TransNetV2(device="cuda")
except Exception as e:
    print(f"CUDA initialization fallback: {e}. Using CPU...")
    model = TransNetV2(device="cpu")

# Discover all video files case-insensitively (.mp4, .MP4, .mkv)
all_videos = sorted([
    p for p in KAGGLE_INPUT.rglob("*")
    if p.suffix.lower() in [".mp4", ".mkv", ".avi"]
])

print(f"Found total {len(all_videos)} video files under /kaggle/input")
if all_videos:
    print("Sample files found:", [p.name for p in all_videos[:5]])

# Filter by START_VIDEO_ID
video_paths = all_videos
if START_VIDEO_ID:
    video_paths = [p for p in video_paths if p.stem >= START_VIDEO_ID]
if END_VIDEO_ID:
    video_paths = [p for p in video_paths if p.stem <= END_VIDEO_ID]

print(f"\nTotal videos to process (starting from '{START_VIDEO_ID}'): {len(video_paths)}")

# Process videos
for vp in tqdm(video_paths, desc="TransNet GPU Shots"):
    video_name = vp.stem
    out_json = OUTPUT_DIR / f"{video_name}.json"
    
    if out_json.exists():
        continue
        
    try:
        # 1. Run TransNet V2 GPU scene detection
        raw_scenes = model.detect_scenes(vp)
        raw_shots = []
        for s in raw_scenes:
            st = to_seconds(s["start_time"])
            et = to_seconds(s["end_time"])
            raw_shots.append({
                "start_time": st,
                "end_time": et,
                "start_frame": int(s.get("start_frame", 0)),
                "end_frame": int(s.get("end_frame", 0)),
                "duration": et - st
            })
            
        # 2. Smart merge
        final_shots = smart_merge_shots(raw_shots, min_duration=MIN_SHOT_DURATION)
            
        # 3. Construct payload with midpoint keyframes
        output_payload = []
        for shot_id, s in enumerate(final_shots):
            st = round(s["start_time"], 2)
            et = round(s["end_time"], 2)
            mid_pts = round((st + et) / 2.0, 3)
            mid_idx = int((s["start_frame"] + s["end_frame"]) / 2)
            
            output_payload.append({
                "shot_id": shot_id + 1,
                "start_time": st,
                "end_time": et,
                "duration": round(s["duration"], 2),
                "n_keyframes": 1,
                "keyframes": [{
                    "n": 1,
                    "pts_time": mid_pts,
                    "fps": 25.0,
                    "frame_idx": mid_idx,
                    "image": f"{mid_idx:06d}.jpg"
                }]
            })
            
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(output_payload, f, ensure_ascii=False, indent=2)
            
    except Exception as e:
        print(f"Error processing {video_name}: {e}")

In [ ]:
# Zip results for 1-click download
generated_jsons = list(OUTPUT_DIR.glob("*.json"))
print(f"Successfully generated {len(generated_jsons)} shot boundary files.")

shutil.make_archive("/kaggle/working/shot_boundaries", "zip", OUTPUT_DIR)
print("Created download package: /kaggle/working/shot_boundaries.zip")